In [ ]:
"""The merchant_revenue pipeline reads 50 parquet files from the silver layer (20 million rows), runs a groupBy('merchant_id') 
aggregation to compute daily revenue, and writes 4 compact files to the gold layer. Two ways to reduce partitions — but only one is correct here"""
#Scenario 6 — Repartition vs Coalesce: The Hidden Trap
#==>Before — Wrong Choice
from pyspark.sql import SparkSession
from pyspark.sql.functions import broadcast
from pyspark.sql.functions import *

spark = SparkSession.builder \
    .appName("Scenario_6")\
    .config("spark.sql.shuffle.partitions", "50") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("error")
#transaction
df_merchant_revenue = spark.read.parquet(r"D:\data engineer\pyspark_practice\pyspark_scenarios\dataset1_transactions\transactions")

# After groupBy, Spark has 50 shuffle partitions
df_merchant_revenue = df_merchant_revenue \
    .groupBy('merchant_id') \
    .agg(sum('amount').alias('daily_revenue'))

# Wrong: triggers a full second shuffle on already-aggregated data
df_merchant_revenue.repartition(4) \
    .write.mode('overwrite') \
    .parquet('data/gold/merchant_revenue/')

In [2]:
#==>After — Correct Choice

df_merchant_revenue = spark.read.parquet(r"D:\data engineer\pyspark_practice\pyspark_scenarios\dataset1_transactions\transactions")
df_merchant_revenue = df_merchant_revenue \
    .groupBy('merchant_id') \
    .agg(sum('amount').alias('daily_revenue'))

# Correct: merges nearby partitions locally, zero network movement
df_merchant_revenue.coalesce(4) \
    .write.mode('overwrite') \
    .parquet('data/gold/merchant_revenue/')

"""Spark simply merges nearby partitions that already sit on the same executor. Partitions 0–12 collapse into file 1, 13–24 into file 2, and so on.
 No data crosses the network. No Exchange node in the physical plan. Shuffle write is 0 GB. Output is 4 identical parquet files with the same rows — 
 just written faste"""

'Spark simply merges nearby partitions that already sit on the same executor. Partitions 0–12 collapse into file 1, 13–24 into file 2, and so on.\n No data crosses the network. No Exchange node in the physical plan. Shuffle write is 0 GB. Output is 4 identical parquet files with the same rows — \n just written faste'